In [ ]:
import numpy as np
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt

from pathlib import Path
from IPython.display import clear_output
from seppy.util import resample_df
from solo_epd_loader import epd_load
from matplotlib.colors import LogNorm

from pad_imputation import (
    convert_to_bool_coverage, load_wind_event, coverage_overlap, 
    load_random_file, intensity_histogram, solo, wind
)

### Generate coverages

In [ ]:
import logging
filehandler = logging.FileHandler(filename="./logs/solo_cov.log", encoding="utf-8")
streamhandler = logging.StreamHandler(sys.stdout)
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', datefmt='%m/%d/%Y %H:%M:%S', handlers=[filehandler, streamhandler], force=True)

bin_width = 4
time_avg = "5min"
time_avg_int = int(time_avg.removesuffix("min"))
root_path = Path(".")
data_path = root_path / "data" / "sc"
save_path = Path(".") / "data" / "coverages" / f"{time_avg_int}min_{bin_width}deg"

data_path.mkdir(exist_ok=True)
save_path.mkdir(exist_ok=True)

max_days = (solo.mission_end - solo.mission_start).days

try:
    start = int(sys.argv[1])
except (IndexError, ValueError):
    start = 567

logging.info(f"Run start: {start}")

for i in range(start, max_days):
    logging.info(f"Day {i}")
    
    date = solo.mission_start + dt.timedelta(days=i)
    logging.info(f"Forming SolO coverages from date {date}")
    try:
        df, df_rtn, df_hci, energies_dict, metadata_dict = epd_load("ept", startdate=date, 
                                                                    level="l3", autodownload=True,
                                                                    path=str(data_path), pos_timestamp="start")

        if (len(df.index) // 60) < 12:
            logging.info(f"Day {i}: Data file not long enough to generate 12 hour coverage, skipping...")
            continue
        
        # if more than 20 % of data is missing, skip
        if np.mean(np.isnan(df.to_numpy())) > 0.20:
            logging.info(f"Day {i}: Over 20 % of the data is missing, skipping...")
            continue

        df = resample_df(df, resample=time_avg, pos_timestamp="start")
        ind = pd.MultiIndex.from_product([solo.sectors, ["min", "center", "max"]])
        df_cov = pd.DataFrame(index=df.index, columns=ind)

        for direction, pa_col, pa_sigma_col in zip(solo.sectors, solo.pitch_angle_mu_columns, 
                                                    solo.pitch_angle_sigma_columns):
            df_cov[(direction, "center")] = df[pa_col]
            df_cov[(direction, "min")] = df[pa_col] - df[pa_sigma_col]
            df_cov[(direction, "max")] = df[pa_col] + df[pa_sigma_col]

        X1, Y1, solo_cov = convert_to_bool_coverage(df_cov, sc=solo, bin_width_deg=bin_width)

        # Generate 4 random 12 hour coverages
        
        for j in range(1, 5):
            random_start = np.random.randint(0, len(solo_cov) - 12*60//time_avg_int)
            cov = solo_cov[random_start:random_start+12*60//time_avg_int]
            logging.info(f"Generated coverage with shape {cov.shape}")
            np.save(save_path / f"{date.strftime("%Y%m%d")}_{time_avg_int}min_{bin_width}deg_{j}", cov)

    except UnboundLocalError:
        logging.info(f"No data was found, skipping...")

    clear_output()

### Generate full and reduced intensity histograms

In [ ]:
data_path = Path("./data/intensities")
cov_path = Path("./data/coverages")
event_path = Path("./data/event_lists")
df = pd.read_csv(event_path / "wind_events_2011.csv", delimiter=",")

energy_channel = 3
bin_width_deg = 4
time_avg_min = 5

hists = []
reduced_hists = []

df_ = df[77:]

for index, data in df_.iterrows():
    solo_cov = load_random_file(cov_path / f"{time_avg_int}min_{bin_width}deg")
    
    onset_dt = pd.to_datetime(df.iloc[index,1]) + pd.to_timedelta(df.iloc[index,2])

    start = onset_dt - pd.Timedelta(hours=2)
    end = onset_dt + pd.Timedelta(hours=10)

    wind_event = load_wind_event(df, str(data_path), "Wind 3DP", start, end, "e", channels=energy_channel, averaging=f"{time_avg_min}min")
    wind_cov = wind_event.coverage
    X, Y, wind_cov_bool = convert_to_bool_coverage(wind_cov, wind, bin_width_deg=bin_width_deg)

    #cov_arr = coverage_overlap(solo_cov, wind_cov_bool)
    
    hist = intensity_histogram("Wind", wind_event.I_data, wind_cov, bin_width_deg).T
    #reduced_hist = np.where(cov_arr == True, hist, np.nan)

    #print(hist.shape, reduced_hist.shape)
    hists.append(hist)
    #reduced_hists.append(reduced_hist)
    
    # TODO metadata (spacecraft, instrument, averaging, channels etc.)

In [ ]:
for i, hist in enumerate(hists):
    np.save(f"hists/hist{i+77}.npy", hist, allow_pickle=False)